<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Functions" data-toc-modified-id="Functions-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Functions</a></span><ul class="toc-item"><li><span><a href="#Helper-Functions" data-toc-modified-id="Helper-Functions-1.1"><span class="toc-item-num">1.1&nbsp;&nbsp;</span>Helper Functions</a></span></li><li><span><a href="#Isolating-Telomeric-Reads-from-NGS" data-toc-modified-id="Isolating-Telomeric-Reads-from-NGS-1.2"><span class="toc-item-num">1.2&nbsp;&nbsp;</span>Isolating Telomeric Reads from NGS</a></span></li><li><span><a href="#Telomeric-Read-Composition" data-toc-modified-id="Telomeric-Read-Composition-1.3"><span class="toc-item-num">1.3&nbsp;&nbsp;</span>Telomeric Read Composition</a></span></li><li><span><a href="#Telomeric-Reads-(Non-WT-or-MT-Analysis)" data-toc-modified-id="Telomeric-Reads-(Non-WT-or-MT-Analysis)-1.4"><span class="toc-item-num">1.4&nbsp;&nbsp;</span>Telomeric Reads (Non WT or MT Analysis)</a></span></li><li><span><a href="#Telomerase-Processivity-Calculations" data-toc-modified-id="Telomerase-Processivity-Calculations-1.5"><span class="toc-item-num">1.5&nbsp;&nbsp;</span>Telomerase Processivity Calculations</a></span></li><li><span><a href="#Probability-Modeling-Using-Consecutivity-Statistics" data-toc-modified-id="Probability-Modeling-Using-Consecutivity-Statistics-1.6"><span class="toc-item-num">1.6&nbsp;&nbsp;</span>Probability Modeling Using Consecutivity Statistics</a></span></li><li><span><a href="#Adjacent-Telomeric-Repeats" data-toc-modified-id="Adjacent-Telomeric-Repeats-1.7"><span class="toc-item-num">1.7&nbsp;&nbsp;</span>Adjacent Telomeric Repeats</a></span></li><li><span><a href="#Telomere-Sequence-Visualization" data-toc-modified-id="Telomere-Sequence-Visualization-1.8"><span class="toc-item-num">1.8&nbsp;&nbsp;</span>Telomere Sequence Visualization</a></span></li></ul></li><li><span><a href="#Sample-Analysis" data-toc-modified-id="Sample-Analysis-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>Sample Analysis</a></span><ul class="toc-item"><li><span><a href="#Isolating-Telomeric-Reads-from-NGS" data-toc-modified-id="Isolating-Telomeric-Reads-from-NGS-2.1"><span class="toc-item-num">2.1&nbsp;&nbsp;</span>Isolating Telomeric Reads from NGS</a></span></li><li><span><a href="#Running-Trimmomatic" data-toc-modified-id="Running-Trimmomatic-2.2"><span class="toc-item-num">2.2&nbsp;&nbsp;</span>Running Trimmomatic</a></span></li><li><span><a href="#Isolating-Fully-Telomeric-Reads" data-toc-modified-id="Isolating-Fully-Telomeric-Reads-2.3"><span class="toc-item-num">2.3&nbsp;&nbsp;</span>Isolating Fully Telomeric Reads</a></span></li><li><span><a href="#Telomeric-Read-Composition" data-toc-modified-id="Telomeric-Read-Composition-2.4"><span class="toc-item-num">2.4&nbsp;&nbsp;</span>Telomeric Read Composition</a></span><ul class="toc-item"><li><ul class="toc-item"><li><span><a href="#Telomeric-Reads-(Non-WT-or-MT-analysis)" data-toc-modified-id="Telomeric-Reads-(Non-WT-or-MT-analysis)-2.4.0.1"><span class="toc-item-num">2.4.0.1&nbsp;&nbsp;</span>Telomeric Reads (Non WT or MT analysis)</a></span></li></ul></li></ul></li><li><span><a href="#Telomerase-Processivity-Calculations" data-toc-modified-id="Telomerase-Processivity-Calculations-2.5"><span class="toc-item-num">2.5&nbsp;&nbsp;</span>Telomerase Processivity Calculations</a></span></li><li><span><a href="#Probability-Modeling" data-toc-modified-id="Probability-Modeling-2.6"><span class="toc-item-num">2.6&nbsp;&nbsp;</span>Probability Modeling</a></span></li><li><span><a href="#Adjacent-Telomeric-Repeats" data-toc-modified-id="Adjacent-Telomeric-Repeats-2.7"><span class="toc-item-num">2.7&nbsp;&nbsp;</span>Adjacent Telomeric Repeats</a></span></li><li><span><a href="#Telomere-Sequence-Visualization" data-toc-modified-id="Telomere-Sequence-Visualization-2.8"><span class="toc-item-num">2.8&nbsp;&nbsp;</span>Telomere Sequence Visualization</a></span></li></ul></li></ul></div>

In [ ]:
import dill
dill.load_session("/media/data_01/ndeimler/DKC_MANUSCRIPT/DKC_manuscript.db")

In [197]:
import dill
dill.dump_session("/media/data_01/ndeimler/DKC_MANUSCRIPT/DKC_manuscript.db")

# Functions

In [1]:
import gzip
import numpy as np
import random
import regex as re
import math
import cairo

## Helper Functions

In [8]:
def fastq_dict_gz(fastq):
    """Loads a gzipped fastq file into dictionary where the keys are the first column of
    header line and the values are a list of length 4 corresponding to the four lines
    of the respective fastq entry"""
    
    fastq_dict = {}
    seq_info = []
    header = ''
    with gzip.open(fastq, 'rt') as fastq:
        linecount = 0
        for line in fastq:
            line = line.strip()
            if linecount % 4 == 0:
                if header != '':
                    fastq_dict[header] = seq_info
                    #check for telomeric sequence and add to is_telomeric
                header = line.split(' ')[0]
                seq_info = [line.split(' ')[1]]
            else:
                seq_info.append(line)
            
            linecount += 1
        fastq_dict[header] = seq_info
    return fastq_dict

def fastq_dict(fastq):
    """Loads a gzipped fastq file into dictionary where the keys are the first column of
    header line and the values are a list of length 4 corresponding to the four lines
    of the respective fastq entry"""
    
    fastq_dict = {}
    seq_info = []
    header = ''
    with open(fastq, 'rt') as fastq:
        linecount = 0
        for line in fastq:
            line = line.strip()
            if linecount % 4 == 0:
                if header != '':
                    fastq_dict[header] = seq_info
                    #check for telomeric sequence and add to is_telomeric
                header = line.split(' ')[0]
                seq_info = [line.split(' ')[1]]
            else:
                seq_info.append(line)
            
            linecount += 1
        fastq_dict[header] = seq_info
    return fastq_dict

def rev_complement(seq):
    """Reverse complements a read from NGS data
    DOES NOT ACCOMODATE FOR AMBIGUOUS IUPAC CODES"""
    rev_dict = {'A':'T', 'T':'A', 'C':'G', 'G':'C', 'N':'N'}
    return ''.join([rev_dict[i] for i in seq[::-1]])

def convert_c_strand(read_dict):
    """If a read in read_dict is identified to be a C strand it reverse complements the read 
    so that G telomeric repeats are identified and marks in the comment line that it is C strand
    THIS DOES NOT ALSO CHANGE THE ORDER OF THE QUALITY SCORES"""
    for head in read_dict:
        if read_dict[head][1].count('CTAACC') > 5:
            read_dict[head][1] = rev_complement(read_dict[head][1])
            read_dict[head][2] = 'C'
        else:
            read_dict[head][1] = read_dict[head][1]
            read_dict[head][2] = 'G'
    return read_dict

## Isolating Telomeric Reads from NGS

In [6]:
def isolate_telomeric_reads(fastq_r1, fastq_r2, telo_fastq):
    """Takes two fastq files and isolates read pairs in which one of the two reads 
    contains greater than 7 telomeric repeats (GGTTAG or GTTTA | CTAACC or TAAAC) writing 
    them to telo_fastq.r1.fastq and telo_fastq.r2.fastq"""
    
    r1 = fastq_dict_gz(fastq_r1)
    r2 = fastq_dict_gz(fastq_r2)
    both = 0
    r1_telo = 0
    r2_telo = 0
    neither = 0
    num_unpaired = 0
    with open('{}.r1.fastq'.format(telo_fastq), 'w') as telo_r1, open('{}.r2.fastq'.format(telo_fastq), 'w') as telo_r2, open('{}.unpaired.fastq'.format(telo_fastq), 'w') as unpaired:
        for item in r1:
            if item in r2:
                if r2[item][1].count('GGTTAG') + r2[item][1].count('GTTTA') > 7 and r1[item][1].count('CTAACC') + r1[item][1].count('TAAAC') > 7:  
                    both += 1
                    telo_r1.write('{} {}\n{}\n{}\n{}\n'.format(item, r1[item][0], r1[item][1], r1[item][2], r1[item][3]))
                    telo_r2.write('{} {}\n{}\n{}\n{}\n'.format(item, r2[item][0], r2[item][1], r2[item][2], r2[item][3]))
                elif r1[item][1].count('GGTTAG') + r1[item][1].count('GTTTA') > 7 and r2[item][1].count('CTAACC') + r2[item][1].count('TAAAC') > 7:    
                    both += 1
                    telo_r1.write('{} {}\n{}\n{}\n{}\n'.format(item, r1[item][0], r1[item][1], r1[item][2], r1[item][3]))
                    telo_r2.write('{} {}\n{}\n{}\n{}\n'.format(item, r2[item][0], r2[item][1], r2[item][2], r2[item][3]))
                elif r2[item][1].count('CTAACC') + r2[item][1].count('TAAAC') > 7 or r2[item][1].count('GGTTAG') + r2[item][1].count('GTTTA') > 7:
                    r2_telo += 1
                    telo_r1.write('{} {}\n{}\n{}\n{}\n'.format(item, r1[item][0], r1[item][1], r1[item][2], r1[item][3]))
                    telo_r2.write('{} {}\n{}\n{}\n{}\n'.format(item, r2[item][0], r2[item][1], r2[item][2], r2[item][3]))
                elif r1[item][1].count('CTAACC') + r1[item][1].count('TAAAC') > 7 or r1[item][1].count('GGTTAG') + r1[item][1].count('GTTTA') > 7:
                    r1_telo += 1
                    telo_r1.write('{} {}\n{}\n{}\n{}\n'.format(item, r1[item][0], r1[item][1], r1[item][2], r1[item][3]))
                    telo_r2.write('{} {}\n{}\n{}\n{}\n'.format(item, r2[item][0], r2[item][1], r2[item][2], r2[item][3]))
                else:
                    neither += 1
            else:
                if r1[item][1].count('GGTTAG') + r1[item][1].count('GTTTA') > 7 or r1[item][1].count('CTAACC') + r1[item][1].count('TAAAC') > 7:
                    unpaired.write('{} {}\n{}\n{}\n{}\n'.format(item, r1[item][0], r1[item][1], r1[item][2], r1[item][3]))
                    num_unpaired += 1
        for item in r2:
            if item in r1:
                pass
            else:
                if r2[item][1].count('GGTTAG') + r2[item][1].count('GTTTA') > 7 or r2[item][1].count('CTAACC') + r2[item][1].count('TAAAC') > 7:
                    unpaired.write('{} {}\n{}\n{}\n{}\n'.format(item, r2[item][0], r2[item][1], r2[item][2], r2[item][3]))
                    num_unpaired += 1

    print('Number of Sequences with 7 Telo repeat in Both Reads: {}'.format(both))
    print('Number of Sequences with 7 Telo repeat only in Forward Read: {}'.format(r1_telo))
    print('Number of Sequences with 7 Telo repeat onnly in Reverse Read: {}'.format(r2_telo))
    print('Number of Unpaired Telomeric Sequences: {}'.format(num_unpaired))
    print('Total Number of Reads containing 7 Telo Repeat: {}'.format(both + r1_telo + r2_telo))
    print('Number of reads not containing 7 Telo Repeat: {}'.format(neither))
    
def get_mt_perc(seq):
    mt_nucl = 0
    mt_count = 0
    pattern = re.compile(r"GTTTA(G?)")
    matches = []
    i = 0
    while i < len(seq):
        match = pattern.match(seq, i)
        if match:
            base_match = match.group(0)
            start = match.start()
            end = match.end()

            # Check if extending to that "G" leads to a new GTTTA match
            extended = base_match
            if base_match.endswith("G"):
                # Check if a GTTTA starts from that G
                if seq[end-1:end+4] != "GTTTA" and seq[end-1:end+4] != "GGTTA":
                    mt_nucl += 6
                    mt_count += 1
                    i = end
                    continue
                else:
                    # Don't include the G to avoid overlap
                    mt_nucl += 5
                    mt_count += 1
                    i = end - 1
                    continue
            else:
                mt_nucl += 5
                mt_count += 1
                i = end
        else:
            i += 1
    return mt_count, mt_nucl, mt_nucl / len(seq)

def isolate_reads(r1_file, r2_file):
    """Takes two files (ideally consisting of only reads identified to be telomeric
    for speed purposes) and determines the class of the read. First reads are trimmed
    to the first and last telomeric repeat (GGTTAG or GTTTA) (C strand is converted to G strand)
    Fully telomeric have length >= 80 post trimming and contain length(read)/6-2 telo repeats
    All other reads are partially telomeric (Subtelomeric)
    """
    
    r1_dict = fastq_dict(r1_file)
    r2_dict = fastq_dict(r2_file)
    r1_dict = convert_c_strand(r1_dict)
    r2_dict = convert_c_strand(r2_dict)

    new_r1_dict = {}
    new_r2_dict = {}
    
    for i in r1_dict:
        for j in range(0, len(r1_dict[i][1])-6):
            if r1_dict[i][1][j:j+6] == "GGTTAG" or r1_dict[i][1][j:j+5] == "GTTTA":
                new_r1_dict[i] = [r1_dict[i][0], r1_dict[i][1][j:],
                                  r1_dict[i][2], r1_dict[i][3][j:]]
                break
    
    for i in r2_dict:
        for j in range(0, len(r2_dict[i][1])-6):
            if r2_dict[i][1][j:j+6] == "GGTTAG" or r2_dict[i][1][j:j+5] == "GTTTA":
                new_r2_dict[i] = [r2_dict[i][0], r2_dict[i][1][j:],
                                      r2_dict[i][2], r2_dict[i][3][j:]]
                break

    
    for i in new_r1_dict:
        for j in range(len(new_r1_dict[i][1])-6,-1, -1):
            if new_r1_dict[i][1][j:j+6] == "GGTTAG" or new_r1_dict[i][1][j:j+5] == "GTTTA":
                new_r1_dict[i] = [new_r1_dict[i][0], new_r1_dict[i][1][0:j+6],
                                  new_r1_dict[i][2], new_r1_dict[i][3][0:j+6]]
                break
                
    for i in new_r2_dict:
        for j in range(len(new_r2_dict[i][1])-6,-1, -1):
            if new_r2_dict[i][1][j:j+6] == "GGTTAG" or new_r2_dict[i][1][j:j+6] == "GTTTA":
                new_r2_dict[i] = [new_r2_dict[i][0], new_r2_dict[i][1][0:j+6],
                                  new_r2_dict[i][2], new_r2_dict[i][3][0:j+6]]
                break
    
    all_telo = []
    
    for i in new_r1_dict:            
        #r1_mt = new_r1_dict[i][1].count('GTTTA')
        r1_wt = new_r1_dict[i][1].count('GGTTAG')
        #r2_mt = new_r2_dict[i][1].count('GTTTA')
        r1_mt = get_mt_perc(new_r1_dict[i][1].replace("GGTTAG", "N"))[0]
        
        if len(new_r1_dict[i][1]) >= 80 and r1_mt + r1_wt >= int(len(new_r1_dict[i][1]) / 6) - 2:
            all_telo.append(new_r1_dict[i][1])
        
            
    for i in new_r2_dict:
        r2_wt = new_r2_dict[i][1].count('GGTTAG')
        r2_mt = get_mt_perc(new_r2_dict[i][1].replace("GGTTAG", "N"))[0]
        if len(new_r2_dict[i][1]) >= 80 and r2_mt + r2_wt >= int(len(new_r2_dict[i][1]) / 6) - 2:
            all_telo.append(new_r2_dict[i][1])

    print("Number of Telomeric Reads: {}".format(len(all_telo)))

    return all_telo

## Telomeric Read Composition

In [18]:
  
def bootstrap_mutant(reads, n=1000, r=2000):

    mt_ratios = []
    wt_ratios = []
    contains = []
    for i in range(0, n):
        #randomly select r reads
        mt_bases = 0
        wt_bases = 0
        total_bases = 0
        contains_counter = 0
        for i in range(0, r):
            rand_int = random.randint(0, len(reads)-1)
            rand_read = reads[rand_int]
            total_bases += len(rand_read)
            wt_bases += rand_read.count('GGTTAG') * 6
            teloseq = rand_read.replace("GGTTAG", "N")
            tmp = teloseq.split("GGTTAG")
            for read in tmp:
                if read == "G":
                    wt_bases += 1
            a,b,c = get_mt_perc(teloseq)
            if a > 0:
                contains_counter += 1
            mt_bases += b

        mt_ratios.append(mt_bases/total_bases)
        wt_ratios.append(wt_bases/total_bases)
        contains.append(contains_counter/r)

    print('Proportion of Reads Containing: {}'.format(sum(contains)/len(contains)))
    print('Proportion of Reads Containing SD: {}'.format(np.std(contains)))
    print('Mutant Base Frequency: {}'.format(sum(mt_ratios)/len(mt_ratios)))
    print('Mutant Base Standard Deviation: {}'.format(np.std(mt_ratios)))
    print('Wild Type Base Frequency: {}'.format(sum(wt_ratios)/len(wt_ratios)))
    print('Wild Type Base Standard Deviation: {}'.format(np.std(wt_ratios)))
    
def read_composition(reads):

    mt_bases = 0
    wt_bases = 0
    total_bases = 0
    contains_counter = 0
    for rand_read in reads:
        total_bases += len(rand_read)
        wt_bases += rand_read.count('GGTTAG') * 6
        teloseq = rand_read.replace("GGTTAG", "N")
        a,b,c = get_mt_perc(teloseq)
        if a > 0:
            contains_counter += 1
        mt_bases += b
    print('Proportion of Reads Containing: {}'.format(contains_counter/len(reads)))
    print('Mutant Base Frequency: {}'.format(mt_bases/total_bases))
    print('Wild Type Base Frequency: {}'.format(wt_bases/total_bases))

## Telomeric Reads (Non WT or MT Analysis)

In [6]:
def point_mutation_analysis(reads):
    """Looks at reads (all fully telomeric reads) and plots the distribution of point
    mutations of telomeric repeat (GGTTAG and all single point mutations there of)"""
    
    repeat = list('GGTTAG')
    nuc = ['A', 'T', 'C', 'G']
    perms = {}
    for i in range(0, len(repeat)):
        for nucleo in nuc:
            if nucleo != repeat[i]:
                tmp = []
                for j in repeat:
                    tmp.append(j)
                tmp[i] = nucleo
                perms[''.join(tmp)] = []
    perms[''.join(repeat)] = []

    repeat_frames = {}

    new_perms = []
    for perm in perms:
        for i in range(0, len(perm)-1):
            tmp = perm[i+1:] + perm[0:i+1]
            repeat_frames[''.join(tmp)] = perm
            new_perms.append(''.join(tmp))
    #repeat frames is a dictionary of all possible permutations with the value being the original permutation
    #new_perms is a list of all possible permutations.
    for item in new_perms:
        perms[item] = []

    tmp_perms = {}
    #perms contains one entry for every possible mutant at every possible read frame
    
    for seq in reads:
        for i in perms:
            tmp_perms[i] = 0

        i = 0
        tmp_perms['GGTTAG'] += seq.count('GGTTAG')
        tmp_perms['GTTTAG'] += seq.count('GTTTAG')
        seq = seq.replace('GGTTAG', 'NNNNNN')
        seq = seq.replace('GTTTAG', 'NNNNNN')
        tmp_perms['GTTTAG'] += seq.count('GTTTA')
        seq = seq.replace('GTTTA', 'NNNNN')
        while i <= len(seq) - 6: 
            if seq[i:i+6] in perms:
                tmp_perms[seq[i:i+6]] += 1
                i += 6
            else:
                i += 1
      
        for i in tmp_perms:
            perms[i].append(tmp_perms[i])
                
    # Every perm is a value in repeat_frames pointing to perm
    tmp = set(repeat_frames.values())
    telo_repeats = {}

    for rep in tmp:
        telo_repeats[rep] = []
        for i in range(0, len(perms['GGTTAG'])):
            telo_repeats[rep].append(0)
    
    for rep in perms:
        for i in range(0, len(perms[rep])):
            if rep in telo_repeats:
                telo_repeats[rep][i] += perms[rep][i]
            else:
                telo_repeats[repeat_frames[rep]][i] += perms[rep][i]
                
    sum_dict = {}
    for rep in telo_repeats:
        sum_dict[rep] = sum(telo_repeats[rep])
    freq = sum(sum_dict.values()) - int(sum_dict['GGTTAG']) - int(sum_dict['GTTTAG'])
    print("Frequency of non-WT and non-GTTTAG repeats: {}".format(freq))
    print("Ratio of Occurence of GTTTAG compared to cumulative sum of all other point mutations: {}".format(int(sum_dict['GTTTAG'])/freq))

    for repeat in sum_dict:
        print("\t{}:{}".format(repeat, sum_dict[repeat]))

## Telomerase Processivity Calculations

In [60]:
def bootstrap_consecutive(reads, mt_out_file, wt_out_file, mt2_out_file, wt2_out_file, percentile=False, full_length=False, n=1000, r=3000):

    mt_add = []
    mt_p = []
    wt_add= []
    wt_p = []
    mt_props = []
    wt_props = []
    wt_a = []
    wt_b = []
    mt_a = []
    mt_b = []
    mt_repeat_procs = []
    wt_repeat_procs = []
    
    for i in range(0, n):
        #randomly select r reads
        rand_reads = []
        for i in range(0, r):
            rand_int = random.randint(0, len(reads)-1)
            rand_reads.append(reads[rand_int])
        
        mt_add_i, mt_i, wt_add_i, wt_i, mt_full, wt_full, wt_a_i, wt_b_i, mt_a_i, mt_b_i, mt_repeat_proc, wt_repeat_proc = consecutivity_analysis(rand_reads, percentile, full_length)
        mt_add.append(mt_add_i)
        mt_p.append(mt_i)
        wt_add.append(wt_add_i)
        wt_p.append(wt_i)
        mt_props.append(mt_full)
        wt_props.append(wt_full)
        mt_repeat_procs.append(mt_repeat_proc)
        wt_repeat_procs.append(wt_repeat_proc)
        wt_a.append(wt_a_i)
        wt_b.append(wt_b_i)
        mt_a.append(mt_a_i)
        mt_b.append(mt_b_i)
        
    print('Mutant Probability of Addition: {}'.format(sum(mt_add)/len(mt_add)))
    print('Mutant Addition Standard Deviation: {}'.format(np.std(mt_add)))
    print('Mutant Probability of Processivity: {}'.format(sum(mt_p)/len(mt_p)))
    print('Mutant Processivity Standard Deviation: {}'.format(np.std(mt_p)))
    
    print('Wild Type Probability of Addition: {}'.format(sum(wt_add)/len(wt_add)))
    print('Wild Type Addition Standard Deviation: {}'.format(np.std(wt_add)))
    print('Wild Type Probability of Processivity: {}'.format(sum(wt_p)/len(wt_p)))
    print('Wild Type Processivity Standard Deviation: {}'.format(np.std(wt_p)))
    
    with open(mt_out_file, 'w') as mt_out:
        for boot_dict in mt_props:
            for length in boot_dict:
                mt_out.write("{}\t".format(boot_dict[length]))
            mt_out.write('\n')
        
    with open(wt_out_file, 'w') as wt_out:
        for boot_dict in wt_props:
            for length in boot_dict:
                wt_out.write("{}\t".format(boot_dict[length]))
            wt_out.write('\n')
            
    with open(mt2_out_file, 'w') as mt_out:
        for iteration in mt_repeat_procs:
            for value in iteration:
                mt_out.write("{}\t".format(value))
            mt_out.write('\n')
        
    with open(wt2_out_file, 'w') as wt_out:
        for iteration in wt_repeat_procs:
            for value in iteration:
                wt_out.write("{}\t".format(value))
            wt_out.write('\n')
            
    print('Wt Slope: {}'.format(sum(wt_a)/len(wt_a)))
    print('WT Intercept: {}'.format(sum(wt_b)/len(wt_b)))
    print("MT Slope: {}".format(sum(mt_a)/len(mt_a)))
    print("MT Intercept: {}".format(sum(mt_b)/len(mt_b)))
    
    return sum(mt_add)/len(mt_add), sum(wt_add)/len(wt_add)
    
def consecutivity_analysis(full, mt_out_file, wt_out_file):
    """Analyzes the reads passed in through full for mutant and wt consecutivity
    returning appropriate statistics and rough figures"""
    start_mt = 0
    end_mt = 0
    start_wt = 0
    end_wt = 0
    total_count = 0
    
    mt_full = {}
    wt_full = {}
    wt_non_complete = {}
    for i in range(1, 26):
        mt_full[i] = 0
        wt_full[i] = 0
        wt_non_complete[i] = 0
    
    for read in full:
        if 'GGTTAG' in read and 'GTTTA' in read:
            total_count += 1
            if read.startswith("GGTTAG"):
                start_wt += 1
            elif read.startswith("GTTTA"):
                start_mt += 1
            if read.endswith('GGTTAG'):
                end_wt += 1
            elif read.endswith('GTTTAG') or read.endswith('GTTTA'):
                end_mt += 1
                
        mt_consec = read_mutant_analysis(read)
        full_length = True
        wt_consec = read_wt_analysis(read, full_length)

        for i in wt_consec:
            wt_full[i] += 1
            for i in wt_consec:
                wt_non_complete[i] += 1
                
        for i in mt_consec:
            mt_full[i] += 1

    sum_mt_full = sum([mt_full[i] for i in mt_full])
    
#     if plots:
#         print("Number of Reads: {}".format(len(full)))
#         print(sum_mt_full)
#         for i in range(1, 26):
#             print("{} repeats: {}".format(i, mt_full[i]/sum_mt_full))

#         for i in range(1,26):
#             if mt_full[i] > 0:
#                 print("{}:{}".format(i,math.log(mt_full[i]/sum_mt_full*100)))
#     #find line of best fit
    mt_a_i, mt_b_i = np.polyfit( [ x for x in mt_full if mt_full[x] > 0], [ math.log(mt_full[x]/sum_mt_full * 100) for x in mt_full if mt_full[x] > 0], 1)

#     if plots:
#         print('{}*x + {}'.format(mt_a_i, mt_b_i))
    mt_p_add_i = pow(math.e, mt_a_i)
#     if plots:
#         print("P(add) = {}".format(mt_p_add_i))
    mt_p_i = (pow(math.e,mt_a_i) - 0.5) / 0.5
                

    percentile = 30

    sum_wt_full = sum([ wt_full[i] for i in wt_full if i < percentile])
    
    wt_a_i, wt_b_i = np.polyfit( [ x for x in wt_full if wt_full[x] > 0 and x < percentile ], [ math.log(wt_full[x]/sum_wt_full * 100) for x in wt_full if wt_full[x] > 0 and x < percentile ], 1)

    wt_p_add_i = pow(math.e, wt_a_i)
    wt_p_i = (pow(math.e,wt_a_i) - 0.5) / 0.5

    mt_sum = sum([mt_full[x] * x  for x in mt_full])
    wt_sum = sum([wt_full[x] * x for x in wt_full])
    
    mt_repeat_proc = [ (mt_full[x] * x) / mt_sum for x in range(1,26)]
    wt_repeat_proc = [ (wt_full[x] * x) / wt_sum  for x in range(1,26)]
    
    print(mt_full)

    print(mt_repeat_proc)
    with open(mt_out_file, 'w') as mt_out:
        for length in mt_full:
            mt_out.write("{}\t{}\n".format(length, mt_full[length]))

    with open(wt_out_file, 'w') as wt_out:
        for length in wt_full:
            wt_out.write("{}\t{}\n".format(length, wt_full[length]))
            
    print('Wt Slope: {}'.format(wt_a_i))
    print('WT Intercept: {}'.format(wt_b_i))
    print("MT Slope: {}".format(mt_a_i))
    print("MT Intercept: {}".format(mt_b_i))
    print("WT Consecutivity: {}".format(wt_p_add_i))
    print("WT Processivity: {}".format(wt_p_i))
    print("MT Consecutivity: {}".format(mt_p_add_i))
    print("MT Processivity: {}".format(mt_p_i))
    print("less than 4 WT: {}".format(sum([wt_full[i] * i for i in wt_full if i < 4])/ sum([wt_full[i] * i for i in wt_full])))
    print("less than 4 MT: {}".format(sum([mt_full[i] * i for i in mt_full if i < 4])/ sum([mt_full[i] * i for i in mt_full])))
    print("greater than 10 WT: {}".format(sum([wt_full[i] * i for i in wt_full if i >= 10])/ sum([wt_full[i] * i for i in wt_full])))
    print("greater than 4 MT: {}".format(sum([mt_full[i] * i for i in mt_full if i >= 10])/ sum([mt_full[i] * i for i in mt_full])))
    
def read_mutant_analysis(read):
    """Splits given read by WT sequences to determine MT consecutivity"""
    #split read by WT
    #check length and appropriate number of repeats
    tmp_list = []
    read = read.split("GGTTAG")
    for subread in read:
        if subread == "":
            continue
        if get_mt_perc(subread)[2] > 0.60:
            consecutive = subread.count('GTTTA')
            if consecutive >= int(len(subread) / 6):
                tmp_list.append(int(consecutive))
    return tmp_list

def read_wt_analysis(read, include_full_length=False):
    """Splits given read by MT sequences to determine WT consecutivity"""
    #split read by WT
    #check length and appropriate number of repeats
    tmp_list = []
    if read.count("GTTTA") < 2 and not include_full_length:
    #if "GTTTA" not in read and not include_full_length:
        return tmp_list
    
    split_read = read.split("GTTTA")
    for subread in split_read[1:-1]:
        if subread == "":
            continue                
        elif subread.count("GGTTAG")*6/len(subread) > 0.60:
            consecutive = subread.count('GGTTAG')
            if consecutive >= int(len(subread) / 6):
                tmp_list.append(int(consecutive))
                
    return tmp_list

## Adjacent Telomeric Repeats

In [9]:
def adjacent_repeat_errors(full):
    """Looks at all possible back to back repeats: MT-MT, WT-WT, MT-WT, WT-MT
    and checks the accuracy of extension looking for additional or deleted nucleotides"""
    wt_insert = 0
    wt_delete = 0
    wt_accurate = 0
    mt_insert = 0
    mt_accurate = 0
    mt_delete = 0
    wt_mt_accurate = 0
    wt_mt_insert = 0
    wt_mt_delete = 0
    mt_wt_insert = 0
    mt_wt_delete = 0
    mt_wt_accurate = 0
    for read in full:
        wt_accurate += len(re.findall('GGTTAGGGTTA', read, overlapped=True))
        #if re.findall('GGTTAG.GGTTAG', read) is not None:
        wt_insert += len(re.findall('GGTTAGGGGTTA', read, overlapped=True))
        #if re.findall('GGTTAGGTTAG', read) is not None:
        wt_delete += len(re.findall('GGTTAGGTTA', read, overlapped=True))
        #if re.findall('GTTTAGTTTA', read) is not None:
        mt_delete += len(re.findall('GTTTAGTTTA', read, overlapped=True))
        mt_accurate += len(re.findall('GTTTAGGTTTA', read, overlapped=True))
        #if re.findall('GTTTAG.GTTTA', read) is not None:
        mt_insert += len(re.findall('GTTTAGGGTTTA', read, overlapped=True))
        wt_mt_accurate += len(re.findall('GGTTAGGTTTA',read,overlapped=True))
        wt_mt_insert += len(re.findall('GGTTAGGGTTTA', read, overlapped=True))
        wt_mt_delete += len(re.findall('GGTTAGTTTA', read, overlapped=True))
        mt_wt_accurate += len(re.findall('GTTTAGGGTTA', read, overlapped=True))
        mt_wt_insert += len(re.findall('GTTTAGGGGTTA', read, overlapped=True))
        mt_wt_delete += len(re.findall('GTTTAGGTTA', read, overlapped=True))
    
    wt_total = wt_accurate + wt_insert + wt_delete
    mt_total = mt_accurate + mt_insert + mt_delete
    wt_mt_total = wt_mt_accurate + wt_mt_insert + wt_mt_delete
    mt_wt_total = mt_wt_accurate + mt_wt_insert + mt_wt_delete
    
    print("Number of adjacent WT Sequences: {}".format(wt_total))
    print("Number of accurate WT adjacent: {}, {}".format(wt_accurate, wt_accurate/wt_total * 100))
    print("Number of inserts WT adjacent: {}, {}".format(wt_insert, wt_insert/wt_total * 100))
    print("Number of deletions WT adjacent: {}, {}".format(wt_delete, wt_delete/wt_total * 100))
    
    print("Number of adjacent MT Sequences: {}".format(mt_total))
    print("Number of accurate MT adjacent: {}, {}".format(mt_accurate, mt_accurate/mt_total * 100))
    print("Number of inserts MT adjacent: {}, {}".format(mt_insert, mt_insert/mt_total * 100))
    print("Number of deletions of MT adjance: {}, {}".format(mt_delete, mt_delete/mt_total * 100))
    
    print("Number of adjacent WT-MT Sequences: {}".format(wt_mt_total))
    print("Number of accurate WT-MT adjacent: {}, {}".format(wt_mt_accurate, wt_mt_accurate/wt_mt_total * 100))
    print("Number of inserts WT-MT adjacent: {}, {}".format(wt_mt_insert, wt_mt_insert/wt_mt_total * 100))
    print("Number of deletions WT-MT adjacent: {}, {}".format(wt_mt_delete, wt_mt_delete/wt_mt_total * 100))

    print("Number of adjacent MT-WT Sequences: {}".format(mt_wt_total))
    print("Number of accurate MT-WT adjacent: {}, {}".format(mt_wt_accurate, mt_wt_accurate/mt_wt_total * 100))
    print("Number of inserts MT-WT adjacent: {}, {}".format(mt_wt_insert, mt_wt_insert/mt_wt_total * 100))
    print("Number of deletions MT-WT adjacent: {}, {}".format(mt_wt_delete, mt_wt_delete/mt_wt_total * 100))

## Telomere Sequence Visualization

In [11]:
def mutant_plot_vis(reads, file_path, read_count=100, save=True):
    """Takes reads and randomly selects read_count number of reads.  
    Sorts them longest to shortest and plots them via pycairo. Green indicates
    WT repeat, Red is mutant of interest and white is everything else. If save will    save a png to file_path"""
    if read_count > len(reads):
        print("Read Count can not be greater than the number of reads")
        return
    
    rnd = np.random.randint(0, len(reads), read_count)
    tmp = []
    for item in rnd:
        tmp.append(reads[item])
        
    tmp_data = sorted(tmp, key=len, reverse=True)
    #height must be number of records *1.52

    surface = cairo.PDFSurface(file_path, 180, 152)
    ctx = cairo.Context(surface)
    ctx.select_font_face("Courier",
                          cairo.FONT_SLANT_NORMAL,
                          cairo.FONT_WEIGHT_NORMAL)
    ctx.set_font_size(2)
    a,b,c,height_txt,d,e = ctx.text_extents('G')
    ctx.rectangle(0, 0, 180, 152)
    ctx.set_source_rgb(1,1,1)
    ctx.fill()

    # Drawing code
    ctx.set_source_rgb(1, 0, 0)
    #for loop goes here
    x_rect = 0
    y_rect = 0
    x_txt = 0
    y_txt = height_txt

    for read in tmp_data:
        i = 0
        while i + 6 <= len(read):
            if read[i:i+6] == 'GGTTAG':
                #print('GGTTAG')
                a,b,width,height,c,d = ctx.text_extents('GGTTAG')
                ctx.set_source_rgb(0.118, -.533,0.898)
                ctx.rectangle(x_rect, y_rect, width, height)
                ctx.move_to(x_txt, y_txt)
                ctx.text_path('GGTTAG')
                ctx.set_fill_rule(cairo.FILL_RULE_EVEN_ODD)
                ctx.fill()
                x_txt += width
                x_rect += width
                i += 6
            elif read[i:i+12] == 'GGTTAGGGGTTA':
                    a,b,width,height,c,d = ctx.text_extents('GGTTAGG')
                    ctx.set_source_rgb(0.847,0.106, 0.376)
                    ctx.rectangle(x_rect, y_rect, width, height)
                    ctx.move_to(x_txt,y_txt)
                    ctx.text_path('GGTTAGG')
                    ctx.set_fill_rule(cairo.FILL_RULE_EVEN_ODD)
                    ctx.fill()
                    x_txt += width
                    x_rect += width
                    i += 7
            elif read[i:i+5] == 'GTTTA':
                if read[i:i+10] == 'GTTTAGGTTA' or read[i:i+8] == 'GTTTAGTT':
                   # print('GTTTA')
                    a,b,width,height,c,d = ctx.text_extents('GTTTA')
                    ctx.set_source_rgb(0.847,0.106, 0.376)
                    ctx.rectangle(x_rect, y_rect, width, height)
                    ctx.move_to(x_txt,y_txt)
                    ctx.text_path('GTTTA')
                    ctx.set_fill_rule(cairo.FILL_RULE_EVEN_ODD)
                    ctx.fill()
                    x_txt += width
                    x_rect += width
                    i+=5
                elif read[i:i+11] == 'GTTTAGGGTTT':
                    a,b,width,height,c,d = ctx.text_extents('GTTTAGG')
                    ctx.set_source_rgb(0.847,0.106, 0.376)
                    ctx.rectangle(x_rect, y_rect, width, height)
                    ctx.move_to(x_txt,y_txt)
                    ctx.text_path('GTTTAGG')
                    ctx.set_fill_rule(cairo.FILL_RULE_EVEN_ODD)
                    ctx.fill()
                    x_txt += width
                    x_rect += width
                    i += 7
                elif read[i:i+6] == 'GTTTAG':
                    a,b,width,height,c,d = ctx.text_extents('GTTTAG')
                    ctx.set_source_rgb(0.847,0.106, 0.376)
                    ctx.rectangle(x_rect, y_rect, width, height)
                    ctx.move_to(x_txt,y_txt)
                    ctx.text_path('GTTTAG')
                    ctx.set_fill_rule(cairo.FILL_RULE_EVEN_ODD)
                    ctx.fill()
                    x_txt += width
                    x_rect += width
                    i += 6
                else:
                    #print('GTTTA')
                    a,b,width,height,c,d = ctx.text_extents('GTTTA')
                    ctx.set_source_rgb(0.847,0.106, 0.376)
                    ctx.rectangle(x_rect, y_rect, width, height)
                    ctx.move_to(x_txt,y_txt)
                    ctx.text_path('GTTTA')
                    ctx.set_fill_rule(cairo.FILL_RULE_EVEN_ODD)
                    ctx.fill()
                    x_txt += width
                    x_rect += width
                    i +=5
            else:
                #print('N')
                a,b,width,height,c,d = ctx.text_extents(read[i])
                ctx.set_source_rgb(0, 0.302, 0.251)
                ctx.rectangle(x_rect, y_rect, width, height)
                ctx.move_to(x_txt,y_txt)
                ctx.text_path(read[i])
                ctx.set_fill_rule(cairo.FILL_RULE_EVEN_ODD)
                ctx.fill()
                x_txt += width
                x_rect += width
                i += 1
        else:
            pass
        y_txt += height_txt
        y_rect += height_txt
        x_rect = 0
        x_txt = 0
    # End of drawing code
    surface.finish()

# Sample Analysis

In [4]:
index_wd = "/media/data_01/ndeimler/DKC_PROJECT/INDEX_PATIENT/"
index_fibro_wd = "/media/data_01/ndeimler/DKC_PROJECT/INDEX_F/"
aunt_wd = "/media/data_01/ndeimler/DKC_PROJECT/DKC5/"
older_sister_wd = "/media/data_01/ndeimler/DKC_PROJECT/DKC8/"
twin_sister_blood_wd = "/media/data_01/ndeimler/DKC_PROJECT/DKC6/"
twin_ster_fibro_wd = "/media/data_01/ndeimler/DKC_PROJECT/DKC6_REAL/"
mother_wd = "/media/data_01/ndeimler/DKC_PROJECT/DKC9/"
cousin_wd = "/media/data_01/ndeimler/DKC_PROJECT/DKC10"
control1262_wd = "/media/data_01/ndeimler/DKC_PROJECT/C_1262_B/"
control1262_fibro_wd = "/media/data_01/ndeimler/DKC_PROJECT/C_1262_F/"
control0079_wd = "/media/data_01/ndeimler/DKC_PROJECT/C_0079_B/"
control0079_fibro_wd = "/media/data_01/ndeimler/DKC_PROJECT/C_0079_F/"

## Isolating Telomeric Reads from NGS

In [14]:
## DKC3 - Index Patient
isolate_telomeric_reads('/media/data_01/shared/SEQUENCING_MAINZ/DKC_SEQUENCING/0005-22_S27_R1_001.fastq.gz'.format(index_wd),
                                        '/media/data_01/shared/SEQUENCING_MAINZ/DKC_SEQUENCING/0005-22_S27_R2_001.fastq.gz'.format(index_wd),
                                        '{}/SEQUENCING_DATA/TELOMERIC/telo_seq'.format(index_wd))

Number of Sequences with 7 Telo repeat in Both Reads: 9671
Number of Sequences with 7 Telo repeat only in Forward Read: 1838
Number of Sequences with 7 Telo repeat onnly in Reverse Read: 1763
Number of Unpaired Telomeric Sequences: 0
Total Number of Reads containing 7 Telo Repeat: 13272
Number of reads not containing 7 Telo Repeat: 255856903


In [15]:
## DKC3 - Index Patient - Fibroblasts
isolate_telomeric_reads('/media/data_01/shared/SEQUENCING_MAINZ/DKC_SEQUENCING//DKC3_S8_R1_001.fastq.gz'.format(index_fibro_wd),
                                        '/media/data_01/shared/SEQUENCING_MAINZ/DKC_SEQUENCING//DKC3_S8_R2_001.fastq.gz'.format(index_fibro_wd),
                                        '{}/SEQUENCING_DATA/TELOMERIC/telo_seq'.format(index_fibro_wd))

Number of Sequences with 7 Telo repeat in Both Reads: 10937
Number of Sequences with 7 Telo repeat only in Forward Read: 1857
Number of Sequences with 7 Telo repeat onnly in Reverse Read: 1557
Number of Unpaired Telomeric Sequences: 0
Total Number of Reads containing 7 Telo Repeat: 14351
Number of reads not containing 7 Telo Repeat: 304190787


In [29]:
## DKC5 - Aunt
isolate_telomeric_reads('/media/data_01/shared/SEQUENCING_MAINZ/DKC_SEQUENCING//0293-22_S39_R1_001.fastq.gz'.format(aunt_wd),
                                        '/media/data_01/shared/SEQUENCING_MAINZ/DKC_SEQUENCING//0293-22_S39_R2_001.fastq.gz'.format(aunt_wd),
                                        '{}/SEQUENCING_DATA/TELOMERIC/telo_seq'.format(aunt_wd))

Number of Sequences with 7 Telo repeat in Both Reads: 11349
Number of Sequences with 7 Telo repeat only in Forward Read: 1506
Number of Sequences with 7 Telo repeat onnly in Reverse Read: 1389
Number of Unpaired Telomeric Sequences: 0
Total Number of Reads containing 7 Telo Repeat: 14244
Number of reads not containing 7 Telo Repeat: 223791151


In [17]:
## DKC6 - Twin Sister - Blood - Proxy for Father
isolate_telomeric_reads('/media/data_01/shared/SEQUENCING_MAINZ/DKC_SEQUENCING//DKC6-0122_S15_R1_001.fastq.gz'.format(twin_sister_blood_wd),
                                        '/media/data_01/shared/SEQUENCING_MAINZ/DKC_SEQUENCING//DKC6-0122_S15_R2_001.fastq.gz'.format(twin_sister_blood_wd),
                                        '{}/SEQUENCING_DATA/TELOMERIC/telo_seq'.format(twin_sister_blood_wd))

Number of Sequences with 7 Telo repeat in Both Reads: 11292
Number of Sequences with 7 Telo repeat only in Forward Read: 1672
Number of Sequences with 7 Telo repeat onnly in Reverse Read: 1677
Number of Unpaired Telomeric Sequences: 0
Total Number of Reads containing 7 Telo Repeat: 14641
Number of reads not containing 7 Telo Repeat: 280652723


In [18]:
## DKC6 - Twin Sister - Fibroblasts
isolate_telomeric_reads('/media/data_01/shared/SEQUENCING_MAINZ/DKC_SEQUENCING//DKC6-0122f_S13_R1_001.fastq.gz'.format(twin_ster_fibro_wd),
                                        '/media/data_01/shared/SEQUENCING_MAINZ/DKC_SEQUENCING//DKC6-0122f_S13_R2_001.fastq.gz'.format(twin_ster_fibro_wd),
                                        '{}/SEQUENCING_DATA/TELOMERIC/telo_seq'.format(twin_ster_fibro_wd))

Number of Sequences with 7 Telo repeat in Both Reads: 11368
Number of Sequences with 7 Telo repeat only in Forward Read: 2417
Number of Sequences with 7 Telo repeat onnly in Reverse Read: 2148
Number of Unpaired Telomeric Sequences: 0
Total Number of Reads containing 7 Telo Repeat: 15933
Number of reads not containing 7 Telo Repeat: 460672752


In [20]:
## DKC8 - Older Sister
isolate_telomeric_reads('/media/data_01/shared/SEQUENCING_MAINZ/DKC_SEQUENCING//DKC8-0122_S14_R1_001.fastq.gz'.format(older_sister_wd),
                                        '/media/data_01/shared/SEQUENCING_MAINZ/DKC_SEQUENCING//DKC8-0122_S14_R2_001.fastq.gz'.format(older_sister_wd),
                                        '{}/SEQUENCING_DATA/TELOMERIC/telo_seq'.format(older_sister_wd)

Number of Sequences with 7 Telo repeat in Both Reads: 10241
Number of Sequences with 7 Telo repeat only in Forward Read: 1393
Number of Sequences with 7 Telo repeat onnly in Reverse Read: 1218
Number of Unpaired Telomeric Sequences: 0
Total Number of Reads containing 7 Telo Repeat: 12852
Number of reads not containing 7 Telo Repeat: 240120066


In [21]:
## DKC9 - Mother
isolate_telomeric_reads('/media/data_01/shared/SEQUENCING_MAINZ/DKC_SEQUENCING//DKC9-0122_S16_R1_001.fastq.gz'.format(mother_wd),
                                        '/media/data_01/shared/SEQUENCING_MAINZ/DKC_SEQUENCING//DKC9-0122_S16_R2_001.fastq.gz'.format(mother_wd),
                                        '{}/SEQUENCING_DATA/TELOMERIC/telo_seq'.format(mother_wd))

Number of Sequences with 7 Telo repeat in Both Reads: 14558
Number of Sequences with 7 Telo repeat only in Forward Read: 1686
Number of Sequences with 7 Telo repeat onnly in Reverse Read: 1558
Number of Unpaired Telomeric Sequences: 0
Total Number of Reads containing 7 Telo Repeat: 17802
Number of reads not containing 7 Telo Repeat: 285286924


In [28]:
## DKC10 - Cousin
isolate_telomeric_reads('/media/data_01/shared/SEQUENCING_MAINZ/DKC_SEQUENCING//DKC10-0122_S14_R1_001.fastq.gz'.format(cousin_wd),
                                        '/media/data_01/shared/SEQUENCING_MAINZ/DKC_SEQUENCING//DKC10-0122_S14_R2_001.fastq.gz'.format(cousin_wd),
                                        '{}/SEQUENCING_DATA/TELOMERIC/telo_seq'.format(cousin_wd))

Number of Sequences with 7 Telo repeat in Both Reads: 27555
Number of Sequences with 7 Telo repeat only in Forward Read: 2769
Number of Sequences with 7 Telo repeat onnly in Reverse Read: 2453
Number of Unpaired Telomeric Sequences: 0
Total Number of Reads containing 7 Telo Repeat: 32777
Number of reads not containing 7 Telo Repeat: 385206746


In [23]:
## Control0079 - Blood
isolate_telomeric_reads('/media/data_01/shared/SEQUENCING_MAINZ/DKC_SEQUENCING//0079-22_S2_R1_001.fastq.gz'.format(control0079_wd),
                                        '/media/data_01/shared/SEQUENCING_MAINZ/DKC_SEQUENCING//0079-22_S2_R2_001.fastq.gz'.format(control0079_wd),
                                        '{}/SEQUENCING_DATA/TELOMERIC/telo_seq'.format(control0079_wd))

Number of Sequences with 7 Telo repeat in Both Reads: 16627
Number of Sequences with 7 Telo repeat only in Forward Read: 2180
Number of Sequences with 7 Telo repeat onnly in Reverse Read: 1827
Number of Unpaired Telomeric Sequences: 0
Total Number of Reads containing 7 Telo Repeat: 20634
Number of reads not containing 7 Telo Repeat: 288299555


In [24]:
## Control0079 - Fibroblasts
isolate_telomeric_reads('/media/data_01/shared/SEQUENCING_MAINZ/DKC_SEQUENCING//DKC1_S7_R1_001.fastq.gz'.format(control0079_fibro_wd),
                                        '/media/data_01/shared/SEQUENCING_MAINZ/DKC_SEQUENCING//DKC1_S7_R2_001.fastq.gz'.format(control0079_fibro_wd),
                                        '{}/SEQUENCING_DATA/TELOMERIC/telo_seq'.format(control0079_fibro_wd))

Number of Sequences with 7 Telo repeat in Both Reads: 13239
Number of Sequences with 7 Telo repeat only in Forward Read: 1657
Number of Sequences with 7 Telo repeat onnly in Reverse Read: 1438
Number of Unpaired Telomeric Sequences: 0
Total Number of Reads containing 7 Telo Repeat: 16334
Number of reads not containing 7 Telo Repeat: 308559090


In [25]:
## Control1262 - Blood
isolate_telomeric_reads('/media/data_01/shared/SEQUENCING_MAINZ/DKC_SEQUENCING//1262-16_S1_R1_001.fastq.gz'.format(control1262_wd),
                                        '/media/data_01/shared/SEQUENCING_MAINZ/DKC_SEQUENCING//1262-16_S1_R2_001.fastq.gz'.format(control1262_wd),
                                        '{}/SEQUENCING_DATA/TELOMERIC/telo_seq'.format(control1262_wd))

Number of Sequences with 7 Telo repeat in Both Reads: 15626
Number of Sequences with 7 Telo repeat only in Forward Read: 1421
Number of Sequences with 7 Telo repeat onnly in Reverse Read: 1460
Number of Unpaired Telomeric Sequences: 0
Total Number of Reads containing 7 Telo Repeat: 18507
Number of reads not containing 7 Telo Repeat: 225721535


In [27]:
## Control1262 - Fibroblasts
isolate_telomeric_reads('/media/data_01/shared/SEQUENCING_MAINZ/DKC_SEQUENCING//1262-16_S9_R1_001.fastq.gz'.format(control1262_fibro_wd),
                                        '/media/data_01/shared/SEQUENCING_MAINZ/DKC_SEQUENCING//1262-16_S9_R2_001.fastq.gz'.format(control1262_fibro_wd),
                                        '{}/SEQUENCING_DATA/TELOMERIC/telo_seq'.format(control1262_fibro_wd))

Number of Sequences with 7 Telo repeat in Both Reads: 27957
Number of Sequences with 7 Telo repeat only in Forward Read: 1905
Number of Sequences with 7 Telo repeat onnly in Reverse Read: 1668
Number of Unpaired Telomeric Sequences: 0
Total Number of Reads containing 7 Telo Repeat: 31530
Number of reads not containing 7 Telo Repeat: 329266322


## Running Trimmomatic

In [9]:
%%bash 

cat /media/data_01/ndeimler/DKC_MANUSCRIPT/SLURM/TELOMERIC_TRIM/INDEX_TELO_TRIM.srun

#!/bin/bash

#SBATCH --job-name INDEX_telo_dna_trim
#SBATCH -p slurmpart
#SBATCH -N 1
#SBATCH -n 1
#SBATCH -c 4
#SBATCH --output ../../INDEX_PATIENT/SEQUENCING_DATA/TELOMERIC/INDEX_%j_telo_trim.out
#SBATCH --error ../../INDEX_PATIENT/SEQUENCING_DATA/TELOMERIC/INDEX_%j_telo_trim.err

source /media/data_01/ndeimler/SOFTWARE/anaconda3/etc/profile.d/conda.sh
conda activate trimmomatic

seq_dir="/media/data_01/ndeimler/DKC_MANUSCRIPT/INDEX_PATIENT/SEQUENCING_DATA/TELOMERIC/"
out_dir="/media/data_01/ndeimler/DKC_MANUSCRIPT/INDEX_PATIENT/SEQUENCING_DATA/TELOMERIC/"

/usr/bin/time -v trimmomatic PE -threads 4 -phred33 $seq_dir/telo_seq.r1.fastq $seq_dir/telo_seq.r2.fastq \
	$out_dir/telo_seq.trimmed.r1.fastq $out_dir/R1.unpaired.trimmed.fq $out_dir/telo_seq.trimmed.r2.fastq $out_dir/R2.unpaired.trimmed.fq \
	SLIDINGWINDOW:5:15 MINLEN:80


## Isolating Fully Telomeric Reads

In [9]:
## DKC3 - Index - Blood
index_b_telo = isolate_reads('{}/SEQUENCING_DATA/TELOMERIC/telo_seq.trimmed.r1.fastq'.format(index_wd),
                                  '{}/SEQUENCING_DATA/TELOMERIC/telo_seq.trimmed.r2.fastq'.format(index_wd))

Number of Telomeric Reads: 11012


In [20]:
## DKC3 - Index - Fibroblast
index_f_telo = isolate_reads('{}/SEQUENCING_DATA/TELOMERIC/telo_seq.trimmed.r1.fastq'.format(index_fibro_wd),
                                  '{}/SEQUENCING_DATA/TELOMERIC/telo_seq.trimmed.r2.fastq'.format(index_fibro_wd))

Number of Telomeric Reads: 12144


In [21]:
## DKC5 - Aunt - Blood
aunt_telo = isolate_reads('{}/SEQUENCING_DATA/TELOMERIC/telo_seq.trimmed.r1.fastq'.format(aunt_wd),
                                  '{}/SEQUENCING_DATA/TELOMERIC/telo_seq.trimmed.r2.fastq'.format(aunt_wd))

Number of Telomeric Reads: 14122


In [22]:
## DKC6 - Twin Sister - Blood
twin_b_telo = isolate_reads('{}/SEQUENCING_DATA/TELOMERIC/telo_seq.trimmed.r1.fastq'.format(twin_sister_blood_wd),
                                  '{}/SEQUENCING_DATA/TELOMERIC/telo_seq.trimmed.r2.fastq'.format(twin_sister_blood_wd))

Number of Telomeric Reads: 12788


In [23]:
## DKC6 - Twin Sister - Fibroblast
twin_f_telo  = isolate_reads('{}/SEQUENCING_DATA/TELOMERIC/telo_seq.trimmed.r1.fastq'.format(twin_ster_fibro_wd),
                                  '{}/SEQUENCING_DATA/TELOMERIC/telo_seq.trimmed.r2.fastq'.format(twin_ster_fibro_wd))

Number of Telomeric Reads: 10999


In [24]:
## DKC8 - Older Sister - Blood
older_sister_telo = isolate_reads('{}/SEQUENCING_DATA/TELOMERIC/telo_seq.trimmed.r1.fastq'.format(older_sister_wd),
                                  '{}/SEQUENCING_DATA/TELOMERIC/telo_seq.trimmed.r2.fastq'.format(older_sister_wd))

Number of Telomeric Reads: 12054


In [25]:
## DKC9 - Mother - Blood
mother_telo = isolate_reads('{}/SEQUENCING_DATA/TELOMERIC/telo_seq.trimmed.r1.fastq'.format(mother_wd),
                                  '{}/SEQUENCING_DATA/TELOMERIC/telo_seq.trimmed.r2.fastq'.format(mother_wd))

Number of Telomeric Reads: 18394


In [26]:
## DKC10 - Cousing - Blood
cousin_telo = isolate_reads('{}/SEQUENCING_DATA/TELOMERIC/telo_seq.trimmed.r1.fastq'.format(cousin_wd),
                                  '{}/SEQUENCING_DATA/TELOMERIC/telo_seq.trimmed.r2.fastq'.format(cousin_wd))

Number of Telomeric Reads: 34962


In [12]:
## Control 0079 - Blood
c0079_b_telo = isolate_reads('{}/SEQUENCING_DATA/TELOMERIC/telo_seq.trimmed.r1.fastq'.format(control0079_wd),
                                  '{}/SEQUENCING_DATA/TELOMERIC/telo_seq.trimmed.r2.fastq'.format(control0079_wd))

Number of Telomeric Reads: 19741


In [28]:
## Control 0079 - Fibroblast
c0079_f_telo = isolate_reads('{}/SEQUENCING_DATA/TELOMERIC/telo_seq.trimmed.r1.fastq'.format(control0079_fibro_wd),
                                  '{}/SEQUENCING_DATA/TELOMERIC/telo_seq.trimmed.r2.fastq'.format(control0079_fibro_wd))

Number of Telomeric Reads: 15946


In [19]:
## Control 1262 - Blood
c1262_b_telo = isolate_reads('{}/SEQUENCING_DATA/TELOMERIC/telo_seq.trimmed.r1.fastq'.format(control1262_wd),
                                  '{}/SEQUENCING_DATA/TELOMERIC/telo_seq.trimmed.r2.fastq'.format(control1262_wd))

Number of Telomeric Reads: 19924


In [30]:
## Control 1262 - Fibroblast
c1262_f_telo = isolate_reads('{}/SEQUENCING_DATA/TELOMERIC/telo_seq.trimmed.r1.fastq'.format(control1262_fibro_wd),
                                  '{}/SEQUENCING_DATA/TELOMERIC/telo_seq.trimmed.r2.fastq'.format(control1262_fibro_wd))

Number of Telomeric Reads: 38551


## Telomeric Read Composition

In [32]:
read_composition(index_b_telo)

Proportion of Reads Containing: 0.5062658917544497
Mutant Base Frequency: 0.09514511218378051
Wild Type Base Frequency: 0.8653340215177451


In [33]:
read_composition(index_f_telo)

Proportion of Reads Containing: 0.4483695652173913
Mutant Base Frequency: 0.08483162614544659
Wild Type Base Frequency: 0.8770142099215363


In [34]:
read_composition(aunt_telo)

Proportion of Reads Containing: 0.43315394420053815
Mutant Base Frequency: 0.08513446713571275
Wild Type Base Frequency: 0.8822377862978438


In [35]:
read_composition(twin_b_telo)

Proportion of Reads Containing: 0.10752267751016578
Mutant Base Frequency: 0.00940133558376752
Wild Type Base Frequency: 0.9626559037205548


In [36]:
read_composition(twin_f_telo)

Proportion of Reads Containing: 0.3844894990453678
Mutant Base Frequency: 0.07149227387628165
Wild Type Base Frequency: 0.8860946978875572


In [37]:
read_composition(older_sister_telo)

Proportion of Reads Containing: 0.27617388418782146
Mutant Base Frequency: 0.04730092513552901
Wild Type Base Frequency: 0.9216851372115292


In [38]:
read_composition(mother_telo)

Proportion of Reads Containing: 0.4188865934543873
Mutant Base Frequency: 0.08266692651747888
Wild Type Base Frequency: 0.8858440878953459


In [39]:
read_composition(cousin_telo)

Proportion of Reads Containing: 0.29700818031005094
Mutant Base Frequency: 0.050403217073637686
Wild Type Base Frequency: 0.9189918750178674


In [40]:
read_composition(c0079_b_telo)

Proportion of Reads Containing: 0.11873765260118535
Mutant Base Frequency: 0.008655656665170425
Wild Type Base Frequency: 0.9647349613212002


In [41]:
read_composition(c0079_f_telo)

Proportion of Reads Containing: 0.08171328232785652
Mutant Base Frequency: 0.006227722800945475
Wild Type Base Frequency: 0.9697758791923119


In [42]:
read_composition(c1262_b_telo)

Proportion of Reads Containing: 0.10524994980927524
Mutant Base Frequency: 0.007117109681321029
Wild Type Base Frequency: 0.9696602007277171


In [43]:
read_composition(c1262_f_telo)

Proportion of Reads Containing: 0.06311120334102877
Mutant Base Frequency: 0.004552252967732798
Wild Type Base Frequency: 0.9766881723484512


#### Telomeric Reads (Non WT or MT analysis)

In [147]:
point_mutation_analysis(index_b_telo)

Frequency of non-WT and non-GTTTAG repeats: 4459
Ratio of Occurence of GTTTAG compared to cumulative sum of all other point mutations: 5.592061000224265
	GGTAAG:666
	GGTTAC:87
	GGTTGG:159
	GGTTAT:178
	CGTTAG:128
	GGGTAG:558
	TGTTAG:320
	GATTAG:37
	GTTTAG:24935
	GGCTAG:50
	AGTTAG:32
	GGATAG:160
	GGTTAA:380
	GGTGAG:189
	GCTTAG:65
	GGTTCG:140
	GGTTTG:1253
	GGTCAG:57
	GGTTAG:212821


In [148]:
point_mutation_analysis(index_f_telo)

Frequency of non-WT and non-GTTTAG repeats: 4178
Ratio of Occurence of GTTTAG compared to cumulative sum of all other point mutations: 5.718764959310675
	GGTAAG:488
	GGTTAC:75
	GGTTGG:201
	GGTTAT:211
	CGTTAG:131
	GGGTAG:602
	TGTTAG:368
	GATTAG:41
	GTTTAG:23893
	GGCTAG:52
	AGTTAG:38
	GGATAG:157
	GGTTAA:447
	GGTGAG:151
	GCTTAG:60
	GGTTCG:96
	GGTTTG:1015
	GGTCAG:45
	GGTTAG:231455


In [149]:
point_mutation_analysis(aunt_telo)

Frequency of non-WT and non-GTTTAG repeats: 5063
Ratio of Occurence of GTTTAG compared to cumulative sum of all other point mutations: 5.61505036539601
	GGTAAG:557
	GGTTAC:103
	GGTTGG:162
	GGTTAT:254
	CGTTAG:197
	GGGTAG:903
	TGTTAG:496
	GATTAG:33
	GTTTAG:28429
	GGCTAG:47
	AGTTAG:54
	GGATAG:156
	GGTTAA:449
	GGTGAG:215
	GCTTAG:121
	GGTTCG:97
	GGTTTG:1125
	GGTCAG:94
	GGTTAG:277102


In [150]:
point_mutation_analysis(twin_b_telo)

Frequency of non-WT and non-GTTTAG repeats: 4434
Ratio of Occurence of GTTTAG compared to cumulative sum of all other point mutations: 0.5814163283716735
	GGTAAG:523
	GGTTAC:76
	GGTTGG:174
	GGTTAT:194
	CGTTAG:162
	GGGTAG:842
	TGTTAG:500
	GATTAG:61
	GTTTAG:2578
	GGCTAG:62
	AGTTAG:77
	GGATAG:132
	GGTTAA:444
	GGTGAG:294
	GCTTAG:138
	GGTTCG:125
	GGTTTG:595
	GGTCAG:35
	GGTTAG:273372


In [151]:
point_mutation_analysis(twin_f_telo)

Frequency of non-WT and non-GTTTAG repeats: 4244
Ratio of Occurence of GTTTAG compared to cumulative sum of all other point mutations: 4.1856738925541945
	GGTAAG:491
	GGTTAC:109
	GGTTGG:232
	GGTTAT:191
	CGTTAG:116
	GGGTAG:653
	TGTTAG:340
	GATTAG:68
	GTTTAG:17764
	GGCTAG:64
	AGTTAG:36
	GGATAG:103
	GGTTAA:568
	GGTGAG:269
	GCTTAG:80
	GGTTCG:125
	GGTTTG:746
	GGTCAG:53
	GGTTAG:206191


In [152]:
point_mutation_analysis(older_sister_telo)

Frequency of non-WT and non-GTTTAG repeats: 4124
Ratio of Occurence of GTTTAG compared to cumulative sum of all other point mutations: 3.28176527643065
	GGTAAG:449
	GGTTAC:96
	GGTTGG:155
	GGTTAT:185
	CGTTAG:152
	GGGTAG:668
	TGTTAG:398
	GATTAG:52
	GTTTAG:13534
	GGCTAG:42
	AGTTAG:47
	GGATAG:127
	GGTTAA:517
	GGTGAG:178
	GCTTAG:80
	GGTTCG:62
	GGTTTG:854
	GGTCAG:62
	GGTTAG:248490


In [153]:
point_mutation_analysis(mother_telo)

Frequency of non-WT and non-GTTTAG repeats: 5882
Ratio of Occurence of GTTTAG compared to cumulative sum of all other point mutations: 6.193811628697722
	GGTAAG:678
	GGTTAC:136
	GGTTGG:197
	GGTTAT:299
	CGTTAG:226
	GGGTAG:988
	TGTTAG:574
	GATTAG:49
	GTTTAG:36432
	GGCTAG:57
	AGTTAG:65
	GGATAG:142
	GGTTAA:521
	GGTGAG:202
	GCTTAG:161
	GGTTCG:111
	GGTTTG:1400
	GGTCAG:76
	GGTTAG:365448


In [154]:
point_mutation_analysis(cousin_telo)

Frequency of non-WT and non-GTTTAG repeats: 13095
Ratio of Occurence of GTTTAG compared to cumulative sum of all other point mutations: 3.174112256586483
	GGTAAG:1658
	GGTTAC:304
	GGTTGG:383
	GGTTAT:635
	CGTTAG:564
	GGGTAG:1895
	TGTTAG:1432
	GATTAG:100
	GTTTAG:41565
	GGCTAG:148
	AGTTAG:183
	GGATAG:461
	GGTTAA:875
	GGTGAG:672
	GCTTAG:397
	GGTTCG:318
	GGTTTG:2885
	GGTCAG:185
	GGTTAG:707689


In [155]:
point_mutation_analysis(c0079_b_telo)

Frequency of non-WT and non-GTTTAG repeats: 7678
Ratio of Occurence of GTTTAG compared to cumulative sum of all other point mutations: 0.4786402709038812
	GGTAAG:1496
	GGTTAC:113
	GGTTGG:204
	GGTTAT:281
	CGTTAG:307
	GGGTAG:1126
	TGTTAG:748
	GATTAG:61
	GTTTAG:3675
	GGCTAG:42
	AGTTAG:108
	GGATAG:330
	GGTTAA:464
	GGTGAG:229
	GCTTAG:222
	GGTTCG:157
	GGTTTG:1766
	GGTCAG:24
	GGTTAG:413796


In [156]:
point_mutation_analysis(c0079_f_telo)

Frequency of non-WT and non-GTTTAG repeats: 5188
Ratio of Occurence of GTTTAG compared to cumulative sum of all other point mutations: 0.4130686198920586
	GGTAAG:767
	GGTTAC:117
	GGTTGG:165
	GGTTAT:197
	CGTTAG:229
	GGGTAG:935
	TGTTAG:562
	GATTAG:58
	GTTTAG:2143
	GGCTAG:52
	AGTTAG:87
	GGATAG:236
	GGTTAA:431
	GGTGAG:234
	GCTTAG:135
	GGTTCG:146
	GGTTTG:805
	GGTCAG:32
	GGTTAG:334979


In [157]:
point_mutation_analysis(c1262_b_telo)

Frequency of non-WT and non-GTTTAG repeats: 7232
Ratio of Occurence of GTTTAG compared to cumulative sum of all other point mutations: 0.41910951327433627
	GGTAAG:1540
	GGTTAC:123
	GGTTGG:139
	GGTTAT:262
	CGTTAG:339
	GGGTAG:1051
	TGTTAG:690
	GATTAG:37
	GTTTAG:3031
	GGCTAG:34
	AGTTAG:84
	GGATAG:347
	GGTTAA:378
	GGTGAG:277
	GCTTAG:165
	GGTTCG:98
	GGTTTG:1591
	GGTCAG:77
	GGTTAG:419045


In [158]:
point_mutation_analysis(c1262_f_telo)

Frequency of non-WT and non-GTTTAG repeats: 10996
Ratio of Occurence of GTTTAG compared to cumulative sum of all other point mutations: 0.34157875591124043
	GGTAAG:1689
	GGTTAC:207
	GGTTGG:289
	GGTTAT:498
	CGTTAG:592
	GGGTAG:2054
	TGTTAG:1230
	GATTAG:141
	GTTTAG:3756
	GGCTAG:81
	AGTTAG:252
	GGATAG:600
	GGTTAA:649
	GGTGAG:462
	GCTTAG:270
	GGTTCG:147
	GGTTTG:1718
	GGTCAG:117
	GGTTAG:825264


## Telomerase Processivity Calculations

In [61]:
consecutivity_analysis(index_b_telo, '{}/PROCESSIVITY/illumina.mt.boot.stretches.out'.format(index_wd), 
                      '{}/PROCESSIVITY/illumina.wt.boot.stretches.out'.format(index_wd))

{1: 8016, 2: 2411, 3: 1002, 4: 430, 5: 206, 6: 78, 7: 71, 8: 69, 9: 27, 10: 25, 11: 24, 12: 7, 13: 3, 14: 4, 15: 3, 16: 3, 17: 1, 18: 4, 19: 1, 20: 0, 21: 0, 22: 0, 23: 0, 24: 0, 25: 1}
[0.3768156818502327, 0.22667230761998777, 0.14130588069383726, 0.08085366426926151, 0.048418182672871714, 0.021999717952333943, 0.023362948338269168, 0.02594838527711183, 0.011422930475250318, 0.011751986085648475, 0.01241009730644479, 0.003948667324777887, 0.001833309829361162, 0.0026324448831852583, 0.0021153574954167254, 0.0022563813284445073, 0.0007991350538240963, 0.0033845719926667607, 0.000893150942509284, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0011751986085648475]
Wt Slope: -0.3097994079400742
WT Intercept: 3.5669589881025123
MT Slope: -0.39134633835202
MT Intercept: 2.6697714331480933
WT Consecutivity: 0.7335940946169879
WT Processivity: 0.46718818923397576
MT Consecutivity: 0.6761459402120106
MT Processivity: 0.35229188042402115
less than 4 WT: 0.3034430571320469
less than 4 MT: 0.7447938701640577
greater

In [62]:
consecutivity_analysis(index_f_telo, '{}/PROCESSIVITY/illumina.mt.boot.stretches.out'.format(index_fibro_wd), 
                      '{}/PROCESSIVITY/illumina.wt.boot.stretches.out'.format(index_fibro_wd))

{1: 8708, 2: 2344, 3: 1043, 4: 376, 5: 166, 6: 70, 7: 76, 8: 37, 9: 32, 10: 6, 11: 15, 12: 0, 13: 2, 14: 0, 15: 7, 16: 0, 17: 0, 18: 1, 19: 0, 20: 0, 21: 0, 22: 0, 23: 0, 24: 0, 25: 0}
[0.41927873272665994, 0.2257210265299244, 0.1506572295247725, 0.07241561943280851, 0.039963407000818525, 0.020222446916076844, 0.02561509942703067, 0.014252010207520824, 0.013866820742452693, 0.002888920988010978, 0.00794453271703019, 0.0, 0.0012518657614714237, 0.0, 0.005055611729019211, 0.0, 0.0, 0.0008666762964032933, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Wt Slope: -0.32777636072338484
WT Intercept: 3.63135826446332
MT Slope: -0.5013185381339367
MT Intercept: 3.2676113390911827
WT Consecutivity: 0.720524139185579
WT Processivity: 0.441048278371158
MT Consecutivity: 0.6057314529165073
MT Processivity: 0.21146290583301464
less than 4 WT: 0.32779505647405466
less than 4 MT: 0.7956569887813568
greater than 10 WT: 0.18746930757898184
greater than 4 MT: 0.018007607491935095


In [66]:
consecutivity_analysis(aunt_telo, '{}/PROCESSIVITY/illumina.mt.boot.stretches.out'.format(aunt_wd), 
                      '{}/PROCESSIVITY/illumina.wt.boot.stretches.out'.format(aunt_wd))

{1: 6570, 2: 2359, 3: 1220, 4: 561, 5: 312, 6: 205, 7: 165, 8: 89, 9: 44, 10: 53, 11: 35, 12: 21, 13: 10, 14: 6, 15: 9, 16: 6, 17: 9, 18: 3, 19: 3, 20: 1, 21: 0, 22: 0, 23: 2, 24: 0, 25: 1}
[0.27135304807533456, 0.19486205187510325, 0.15116471171320006, 0.09268131505038824, 0.06443086073021642, 0.050801255575747564, 0.04770361804064101, 0.029406905666611596, 0.01635552618536263, 0.021889971914753014, 0.015901206013547, 0.010408062117958037, 0.0053692383941847015, 0.003469354039319346, 0.0055757475631918055, 0.003964976044936395, 0.00631918057161738, 0.0022302990252767224, 0.0023542045266809848, 0.0008260366760284157, 0.0, 0.0, 0.001899884354865356, 0.0, 0.0010325458450355196]
Wt Slope: -0.3107909707861144
WT Intercept: 3.6234273326756687
MT Slope: -0.3541425595542418
MT Intercept: 2.945887322499246
WT Consecutivity: 0.7328670504832354
WT Processivity: 0.46573410096647083
MT Consecutivity: 0.7017749155047575
MT Processivity: 0.4035498310095149
less than 4 WT: 0.28740642471335165
less th

In [64]:
consecutivity_analysis(twin_f_telo, '{}/PROCESSIVITY/illumina.mt.boot.stretches.out'.format(twin_ster_fibro_wd), 
                      '{}/PROCESSIVITY/illumina.wt.boot.stretches.out'.format(twin_ster_fibro_wd))

{1: 5703, 2: 1658, 3: 738, 4: 325, 5: 207, 6: 64, 7: 48, 8: 24, 9: 36, 10: 13, 11: 3, 12: 1, 13: 3, 14: 4, 15: 4, 16: 1, 17: 1, 18: 0, 19: 0, 20: 0, 21: 0, 22: 0, 23: 0, 24: 0, 25: 0}
[0.37601371398430805, 0.21863255752620822, 0.14597481374035737, 0.08571240192523241, 0.06824025845585811, 0.025318124876376344, 0.0221533592668293, 0.012659062438188172, 0.02136216786444254, 0.008571240192523241, 0.002175776356563592, 0.0007911914023867607, 0.0025713720577569725, 0.00369222654447155, 0.003955957011933803, 0.0010549218698490143, 0.0011208544867145776, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Wt Slope: -0.3222544320917971
WT Intercept: 3.6320773200874896
MT Slope: -0.5145795599801121
MT Intercept: 3.4506669618378822
WT Consecutivity: 0.7245138273083449
WT Processivity: 0.4490276546166898
MT Consecutivity: 0.5977518605975571
MT Processivity: 0.19550372119511428
less than 4 WT: 0.32268647477561124
less than 4 MT: 0.7406210852508736
greater than 10 WT: 0.21993190962550294
greater than 4 MT: 0.0

In [65]:
consecutivity_analysis(mother_telo, '{}/PROCESSIVITY/illumina.mt.boot.stretches.out'.format(mother_wd), 
                      '{}/PROCESSIVITY/illumina.wt.boot.stretches.out'.format(mother_wd))

{1: 8761, 2: 3285, 3: 1623, 4: 757, 5: 465, 6: 253, 7: 157, 8: 121, 9: 67, 10: 39, 11: 23, 12: 11, 13: 9, 14: 7, 15: 9, 16: 5, 17: 2, 18: 7, 19: 0, 20: 1, 21: 0, 22: 0, 23: 0, 24: 1, 25: 0}
[0.28125200642054576, 0.21091492776886037, 0.15630818619582665, 0.09720706260032103, 0.07463884430176565, 0.04873194221508828, 0.035280898876404496, 0.031075441412520066, 0.019357945425361157, 0.012520064205457464, 0.00812199036918138, 0.004237560192616372, 0.003756019261637239, 0.003146067415730337, 0.004333868378812199, 0.002568218298555377, 0.0010914927768860352, 0.004044943820224719, 0.0, 0.0006420545746388443, 0.0, 0.0, 0.0, 0.0007704654895666132, 0.0]
Wt Slope: -0.3233570865502595
WT Intercept: 3.749550376320129
MT Slope: -0.40578477138483127
MT Intercept: 3.1478651774249586
WT Consecutivity: 0.7237153791935317
WT Processivity: 0.44743075838706337
MT Consecutivity: 0.6664535918734631
MT Processivity: 0.3329071837469262
less than 4 WT: 0.2730365474339036
less than 4 MT: 0.6484751203852327
great

## Adjacent Telomeric Repeats

In [35]:
adjacent_repeat_errors(index_b_telo)

Number of adjacent WT Sequences: 186578
Number of accurate WT adjacent: 182410, 97.76608174597219
Number of inserts WT adjacent: 2510, 1.345281866029221
Number of deletions WT adjacent: 1658, 0.8886363879985851
Number of adjacent MT Sequences: 9789
Number of accurate MT adjacent: 8091, 82.65399938706712
Number of inserts MT adjacent: 1496, 15.28245990397385
Number of deletions of MT adjance: 202, 2.0635407089590356
Number of adjacent WT-MT Sequences: 11850
Number of accurate WT-MT adjacent: 6737, 56.85232067510548
Number of inserts WT-MT adjacent: 5110, 43.12236286919831
Number of deletions WT-MT adjacent: 3, 0.025316455696202535
Number of adjacent MT-WT Sequences: 12435
Number of accurate MT-WT adjacent: 10958, 88.1222356252513
Number of inserts MT-WT adjacent: 76, 0.6111781262565339
Number of deletions MT-WT adjacent: 1401, 11.266586248492159


In [51]:
adjacent_repeat_errors(index_f_telo)

Number of adjacent WT Sequences: 203834
Number of accurate WT adjacent: 198981, 97.6191410657692
Number of inserts WT adjacent: 2952, 1.448237291129056
Number of deletions WT adjacent: 1901, 0.9326216431017397
Number of adjacent MT Sequences: 8412
Number of accurate MT adjacent: 6498, 77.24679029957204
Number of inserts MT adjacent: 1716, 20.399429386590583
Number of deletions of MT adjance: 198, 2.353780313837375
Number of adjacent WT-MT Sequences: 12282
Number of accurate WT-MT adjacent: 6548, 53.313792541931285
Number of inserts WT-MT adjacent: 5729, 46.64549747598111
Number of deletions WT-MT adjacent: 5, 0.040709982087607885
Number of adjacent MT-WT Sequences: 12983
Number of accurate MT-WT adjacent: 11519, 88.7237156281291
Number of inserts MT-WT adjacent: 85, 0.6547023030116306
Number of deletions MT-WT adjacent: 1379, 10.621582068859277


In [52]:
adjacent_repeat_errors(aunt_telo)

Number of adjacent WT Sequences: 247845
Number of accurate WT adjacent: 243079, 98.07702394641812
Number of inserts WT adjacent: 3467, 1.3988581573160646
Number of deletions WT adjacent: 1299, 0.5241178962658113
Number of adjacent MT Sequences: 13780
Number of accurate MT adjacent: 12230, 88.75181422351234
Number of inserts MT adjacent: 1346, 9.7677793904209
Number of deletions of MT adjance: 204, 1.4804063860667633
Number of adjacent WT-MT Sequences: 10956
Number of accurate WT-MT adjacent: 6257, 57.110259218692946
Number of inserts WT-MT adjacent: 4698, 42.88061336254108
Number of deletions WT-MT adjacent: 1, 0.009127418765972983
Number of adjacent MT-WT Sequences: 11643
Number of accurate MT-WT adjacent: 10035, 86.18912651378511
Number of inserts MT-WT adjacent: 69, 0.5926307652666839
Number of deletions MT-WT adjacent: 1539, 13.21824272094821


In [53]:
adjacent_repeat_errors(twin_b_telo)

Number of adjacent WT Sequences: 253299
Number of accurate WT adjacent: 249203, 98.38293874038192
Number of inserts WT adjacent: 3862, 1.5246803185168516
Number of deletions WT adjacent: 234, 0.0923809411012282
Number of adjacent MT Sequences: 910
Number of accurate MT adjacent: 104, 11.428571428571429
Number of inserts MT adjacent: 515, 56.59340659340659
Number of deletions of MT adjance: 291, 31.978021978021975
Number of adjacent WT-MT Sequences: 1416
Number of accurate WT-MT adjacent: 593, 41.87853107344633
Number of inserts WT-MT adjacent: 818, 57.7683615819209
Number of deletions WT-MT adjacent: 5, 0.3531073446327684
Number of adjacent MT-WT Sequences: 1551
Number of accurate MT-WT adjacent: 1489, 96.00257898130239
Number of inserts MT-WT adjacent: 52, 3.352675693101225
Number of deletions MT-WT adjacent: 10, 0.6447453255963894


In [54]:
adjacent_repeat_errors(twin_f_telo)

Number of adjacent WT Sequences: 182204
Number of accurate WT adjacent: 176867, 97.07086562314768
Number of inserts WT adjacent: 4204, 2.3073039011218195
Number of deletions WT adjacent: 1133, 0.6218304757304999
Number of adjacent MT Sequences: 6990
Number of accurate MT adjacent: 5353, 76.58082975679542
Number of inserts MT adjacent: 1454, 20.80114449213162
Number of deletions of MT adjance: 183, 2.6180257510729614
Number of adjacent WT-MT Sequences: 8312
Number of accurate WT-MT adjacent: 3990, 48.00288739172281
Number of inserts WT-MT adjacent: 4320, 51.9730510105871
Number of deletions WT-MT adjacent: 2, 0.024061597690086624
Number of adjacent MT-WT Sequences: 8895
Number of accurate MT-WT adjacent: 7747, 87.0938729623384
Number of inserts MT-WT adjacent: 115, 1.2928611579539067
Number of deletions MT-WT adjacent: 1033, 11.6132658797077


In [55]:
adjacent_repeat_errors(older_sister_telo)

Number of adjacent WT Sequences: 226348
Number of accurate WT adjacent: 222433, 98.27036245073957
Number of inserts WT adjacent: 2880, 1.2723770477318113
Number of deletions WT adjacent: 1035, 0.45726050152861963
Number of adjacent MT Sequences: 5829
Number of accurate MT adjacent: 4729, 81.12883856579172
Number of inserts MT adjacent: 903, 15.491507977354606
Number of deletions of MT adjance: 197, 3.379653456853663
Number of adjacent WT-MT Sequences: 5975
Number of accurate WT-MT adjacent: 2964, 49.60669456066945
Number of inserts WT-MT adjacent: 3009, 50.35983263598326
Number of deletions WT-MT adjacent: 2, 0.03347280334728034
Number of adjacent MT-WT Sequences: 6280
Number of accurate MT-WT adjacent: 5485, 87.34076433121018
Number of inserts MT-WT adjacent: 64, 1.019108280254777
Number of deletions MT-WT adjacent: 731, 11.640127388535031


In [56]:
adjacent_repeat_errors(mother_telo)

Number of adjacent WT Sequences: 327535
Number of accurate WT adjacent: 321849, 98.2640023203627
Number of inserts WT adjacent: 3597, 1.0982032454546842
Number of deletions WT adjacent: 2089, 0.6377944341826064
Number of adjacent MT Sequences: 16897
Number of accurate MT adjacent: 14759, 87.34686630762857
Number of inserts MT adjacent: 1772, 10.487068710421967
Number of deletions of MT adjance: 366, 2.166064981949458
Number of adjacent WT-MT Sequences: 14873
Number of accurate WT-MT adjacent: 8208, 55.18725206750488
Number of inserts WT-MT adjacent: 6660, 44.779129967054395
Number of deletions WT-MT adjacent: 5, 0.03361796544073153
Number of adjacent MT-WT Sequences: 15606
Number of accurate MT-WT adjacent: 13340, 85.4799436114315
Number of inserts MT-WT adjacent: 114, 0.7304882737408689
Number of deletions MT-WT adjacent: 2152, 13.78956811482763


In [57]:
adjacent_repeat_errors(cousin_telo)

Number of adjacent WT Sequences: 640236
Number of accurate WT adjacent: 631838, 98.68829619077964
Number of inserts WT adjacent: 5179, 0.8089204605801611
Number of deletions WT adjacent: 3219, 0.5027833486401889
Number of adjacent MT Sequences: 14217
Number of accurate MT adjacent: 11555, 81.275937258212
Number of inserts MT adjacent: 2360, 16.59984525567982
Number of deletions of MT adjance: 302, 2.12421748610818
Number of adjacent WT-MT Sequences: 21373
Number of accurate WT-MT adjacent: 12238, 57.25915875169606
Number of inserts WT-MT adjacent: 9132, 42.72680484723717
Number of deletions WT-MT adjacent: 3, 0.01403640106676648
Number of adjacent MT-WT Sequences: 22541
Number of accurate MT-WT adjacent: 19938, 88.45215385297902
Number of inserts MT-WT adjacent: 106, 0.4702542034514884
Number of deletions MT-WT adjacent: 2497, 11.077591943569495


In [58]:
adjacent_repeat_errors(c0079_b_telo)

Number of adjacent WT Sequences: 383497
Number of accurate WT adjacent: 379023, 98.83336766650065
Number of inserts WT adjacent: 4277, 1.1152629616398564
Number of deletions WT adjacent: 197, 0.051369371859493036
Number of adjacent MT Sequences: 799
Number of accurate MT adjacent: 186, 23.27909887359199
Number of inserts MT adjacent: 453, 56.69586983729662
Number of deletions of MT adjance: 160, 20.02503128911139
Number of adjacent WT-MT Sequences: 2061
Number of accurate WT-MT adjacent: 1408, 68.31635128578361
Number of inserts WT-MT adjacent: 651, 31.58660844250364
Number of deletions WT-MT adjacent: 2, 0.09704027171276079
Number of adjacent MT-WT Sequences: 2450
Number of accurate MT-WT adjacent: 2349, 95.87755102040816
Number of inserts MT-WT adjacent: 84, 3.428571428571429
Number of deletions MT-WT adjacent: 17, 0.6938775510204082


In [59]:
adjacent_repeat_errors(c0079_f_telo)

Number of adjacent WT Sequences: 311815
Number of accurate WT adjacent: 308219, 98.84675208056059
Number of inserts WT adjacent: 3388, 1.0865416994050958
Number of deletions WT adjacent: 208, 0.06670622003431521
Number of adjacent MT Sequences: 533
Number of accurate MT adjacent: 95, 17.823639774859288
Number of inserts MT adjacent: 363, 68.10506566604127
Number of deletions of MT adjance: 75, 14.071294559099437
Number of adjacent WT-MT Sequences: 1237
Number of accurate WT-MT adjacent: 650, 52.54648342764754
Number of inserts WT-MT adjacent: 584, 47.21099434114794
Number of deletions WT-MT adjacent: 3, 0.2425222312045271
Number of adjacent MT-WT Sequences: 1385
Number of accurate MT-WT adjacent: 1308, 94.44043321299638
Number of inserts MT-WT adjacent: 67, 4.837545126353791
Number of deletions MT-WT adjacent: 10, 0.7220216606498195


In [60]:
adjacent_repeat_errors(c1262_b_telo)

Number of adjacent WT Sequences: 389506
Number of accurate WT adjacent: 387301, 99.43389832249055
Number of inserts WT adjacent: 1991, 0.511160290213758
Number of deletions WT adjacent: 214, 0.05494138729570276
Number of adjacent MT Sequences: 589
Number of accurate MT adjacent: 143, 24.27843803056027
Number of inserts MT adjacent: 274, 46.5195246179966
Number of deletions of MT adjance: 172, 29.20203735144312
Number of adjacent WT-MT Sequences: 1717
Number of accurate WT-MT adjacent: 1276, 74.31566686080373
Number of inserts WT-MT adjacent: 441, 25.684333139196276
Number of deletions WT-MT adjacent: 0, 0.0
Number of adjacent MT-WT Sequences: 2165
Number of accurate MT-WT adjacent: 2106, 97.27482678983834
Number of inserts MT-WT adjacent: 47, 2.1709006928406467
Number of deletions MT-WT adjacent: 12, 0.5542725173210161


In [61]:
adjacent_repeat_errors(c1262_f_telo)

Number of adjacent WT Sequences: 772301
Number of accurate WT adjacent: 768378, 99.49203743100165
Number of inserts WT adjacent: 3604, 0.4666574301988473
Number of deletions WT adjacent: 319, 0.04130513879950952
Number of adjacent MT Sequences: 976
Number of accurate MT adjacent: 116, 11.885245901639344
Number of inserts MT adjacent: 542, 55.5327868852459
Number of deletions of MT adjance: 318, 32.58196721311475
Number of adjacent WT-MT Sequences: 2131
Number of accurate WT-MT adjacent: 1456, 68.3247301736274
Number of inserts WT-MT adjacent: 671, 31.48756452369779
Number of deletions WT-MT adjacent: 4, 0.18770530267480057
Number of adjacent MT-WT Sequences: 2511
Number of accurate MT-WT adjacent: 2439, 97.1326164874552
Number of inserts MT-WT adjacent: 61, 2.429311031461569
Number of deletions MT-WT adjacent: 11, 0.43807248108323377


## Telomere Sequence Visualization

In [13]:
mutant_plot_vis(index_b_telo, "{}/SEQUENCING_DATA/TELOMERIC/100_reads_visualization.pdf".format(index_wd), read_count=100, save=True)

In [186]:
mutant_plot_vis(index_f_telo, "{}/SEQUENCING_DATA/TELOMERIC/100_reads_visualization.pdf".format(index_fibro_wd), read_count=100, save=True)

In [187]:
mutant_plot_vis(aunt_telo, "{}/SEQUENCING_DATA/TELOMERIC/100_reads_visualization.pdf".format(aunt_wd), read_count=100, save=True)

In [188]:
mutant_plot_vis(twin_b_telo, "{}/SEQUENCING_DATA/TELOMERIC/100_reads_visualization.pdf".format(twin_sister_blood_wd), read_count=100, save=True)

In [189]:
mutant_plot_vis(twin_f_telo, "{}/SEQUENCING_DATA/TELOMERIC/100_reads_visualization.pdf".format(twin_ster_fibro_wd), read_count=100, save=True)

In [190]:
mutant_plot_vis(older_sister_telo, "{}/SEQUENCING_DATA/TELOMERIC/100_reads_visualization.pdf".format(older_sister_wd), read_count=100, save=True)

In [191]:
mutant_plot_vis(mother_telo, "{}/SEQUENCING_DATA/TELOMERIC/100_reads_visualization.pdf".format(mother_wd), read_count=100, save=True)

In [192]:
mutant_plot_vis(cousin_telo, "{}/SEQUENCING_DATA/TELOMERIC/100_reads_visualization.pdf".format(cousin_wd), read_count=100, save=True)

In [23]:
mutant_plot_vis(c0079_b_telo, "{}/SEQUENCING_DATA/TELOMERIC/100_reads_visualization.pdf".format(control0079_wd), read_count=100, save=True)

In [194]:
mutant_plot_vis(c0079_f_telo, "{}/SEQUENCING_DATA/TELOMERIC/100_reads_visualization.pdf".format(control0079_fibro_wd), read_count=100, save=True)

In [24]:
mutant_plot_vis(c1262_b_telo, "{}/SEQUENCING_DATA/TELOMERIC/100_reads_visualization.pdf".format(control1262_wd), read_count=100, save=True)

In [196]:
mutant_plot_vis(c1262_f_telo, "{}/SEQUENCING_DATA/TELOMERIC/100_reads_visualization.pdf".format(control1262_fibro_wd), read_count=100, save=True)